# Chronological Splits and One-Hour-Ahead Baselines

This notebook explains the fixed UTC evaluation periods and inspects the reproducible outputs created by `src/run_baselines.py`. The canonical feature master is only read; it is never rewritten here.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed' / 'eia_ciso_hourly_features.csv').exists():
    ROOT = ROOT.parent
SOURCE = ROOT / 'data' / 'processed' / 'eia_ciso_hourly_features.csv'
TABLE_DIR = ROOT / 'results' / 'baselines' / 'tables'
FIGURE_DIR = ROOT / 'results' / 'baselines' / 'figures'
assert SOURCE.exists(), SOURCE
assert TABLE_DIR.exists(), 'Run src/run_baselines.py first.'
print(f'Project root: {ROOT}')

## Why chronological splitting is mandatory

Forecasts must imitate time moving forward. Training uses 2022–2023, validation uses the first half of 2024, and test uses the second half of 2024. Random splitting would place later observations in training while scoring earlier observations, letting seasonal patterns and fitted statistics learn from the future. Defining the periods first also keeps missing-value filtering from silently moving a boundary.

In [ ]:
split_summary = pd.read_csv(TABLE_DIR / 'split_summary.csv')
columns = [
    'split', 'start_utc', 'end_utc', 'total_rows', 'target_available_rows',
    'persistence_1h_eligible_rows', 'daily_seasonal_naive_24h_eligible_rows',
    'weekly_seasonal_naive_168h_eligible_rows',
    'train_hour_of_week_mean_eligible_rows'
]
print(split_summary[columns].to_string(index=False))

## What the four baselines predict

- **Persistence:** the demand observed one hour earlier (`demand_lag_1h`).
- **Daily seasonal naive:** the demand at the same UTC hour one day earlier (`demand_lag_24h`).
- **Weekly seasonal naive:** the demand at the same UTC hour one week earlier (`demand_lag_168h`).
- **Hour-of-week mean:** the mean training demand for one of 168 UTC weekday/hour categories.

The hour-of-week lookup is fitted only on available training targets, then frozen. If validation or test contains an unseen category, the training global mean is used and the fallback is counted. Same-hour renewable measurements are not predictors for any baseline.

In [ ]:
model_spec = pd.read_csv(TABLE_DIR / 'baseline_model_specification.csv')
lookup = pd.read_csv(TABLE_DIR / 'hour_of_week_training_lookup.csv')
print(model_spec[['model_name', 'prediction_source', 'fit_split']].to_string(index=False))
print(f'Hour-of-week categories fitted: {len(lookup)}')
print(f"Training observations used: {int(lookup['training_observation_count'].sum()):,}")
print(f"Latest contributing timestamp: {lookup['latest_contributing_timestamp_utc'].max()} UTC")

## Reading the metrics

MAE is the primary ranking metric and is easy to read in MWh. RMSE gives extra weight to large misses. MAPE and sMAPE express relative error, while mean error (bias) shows systematic over-prediction when positive and under-prediction when negative. R² compares squared error with predicting the evaluation-period mean; it can be negative for a weak forecast. Metrics use only rows with an available target and a numeric prediction, without filling missing values.

In [ ]:
metrics = pd.read_csv(TABLE_DIR / 'baseline_metrics_all.csv')
metric_columns = [
    'split', 'mae_rank', 'model_name', 'observation_count', 'mae_mwh',
    'rmse_mwh', 'mape_pct', 'smape_pct', 'mean_error_bias_mwh', 'r_squared'
]
print(metrics[metric_columns].sort_values(['split', 'mae_rank']).round(3).to_string(index=False))

## Error analysis and demand tails

Hourly, monthly, and weekday summaries show when errors are concentrated. The bottom and top 10% thresholds are calculated separately from each evaluation split's observed targets, so test demand does not influence the validation threshold and vice versa. The saved tables are summaries; the notebook deliberately does not print the full timestamp-level prediction files.

In [ ]:
thresholds = pd.read_csv(TABLE_DIR / 'demand_percentile_thresholds.csv')
tails = pd.read_csv(TABLE_DIR / 'demand_percentile_performance.csv')
test_peaks = tails.query("split == 'test' and demand_group == 'top_10_percent'")
print('Split-local thresholds:')
print(thresholds.round(2).to_string(index=False))
print('\nTop-10% test-demand performance:')
print(test_peaks[['model_name', 'observation_count', 'mae_mwh', 'rmse_mwh', 'mape_pct']].sort_values('mae_mwh').round(2).to_string(index=False))

## One-step-ahead is not recursive 24-hour forecasting

This evaluation issues a forecast for hour `t` after the true earlier demand observations have become available. The clock then advances one hour and history is refreshed. In a recursive 24-hour forecast, later horizons cannot use actual demand that occurs after the forecast origin; they must use earlier model predictions instead. Recursive errors can therefore accumulate, so these results are a necessary benchmark rather than a claim about 24-hour performance.

In [ ]:
weeks = pd.read_csv(TABLE_DIR / 'representative_weeks.csv')
figure_names = [
    '01_validation_metrics_comparison.png',
    '02_test_metrics_comparison.png',
    '03_validation_representative_week.png',
    '04_test_representative_week.png',
    '05_test_absolute_error_by_utc_hour.png',
    '06_test_mae_by_calendar_month.png',
    '07_test_error_distribution.png',
    '08_peak_demand_test_performance.png',
]
for name in figure_names:
    image = plt.imread(FIGURE_DIR / name)
    assert image.size > 0
print(weeks.to_string(index=False))
print(f'Verified {len(figure_names)} readable figure files.')

## Reproducibility checks

`src/validate_baselines.py` independently reconstructs split membership, the training-only hour-of-week lookup, every prediction formula, and every metric from the saved prediction rows. It also checks the feature-master SHA-256, the absence of same-hour renewable predictors, and the boundary and missingness rules.